<a href="https://colab.research.google.com/github/srinivas340/ml_practice/blob/main/fake_news_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

About the Dataset:

1. id: unique id for a news article
2. title: the title of a news article
3. author: author of the news article
4. text: the text of the article; could be incomplete
5. label: a label that marks whether the news article is real or fake:

           1: Fake news
           0: real News





In [24]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [25]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [26]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [27]:
df=pd.read_csv('/content/train.csv')

In [28]:
df.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [29]:
df.shape

(20800, 5)

In [30]:
df.isnull().sum()

,0
id,0
title,558
author,1957
text,39
label,0


In [31]:
# replacing the null values with empty string
df= df.fillna('')

In [32]:
#merging the author name and news title
df['content']=df['author']+' '+df['title']
print(df['content'])

0        Darrell Lucus House Dem Aide: We Didn’t Even S...
1        Daniel J. Flynn FLYNN: Hillary Clinton, Big Wo...
2        Consortiumnews.com Why the Truth Might Get You...
3        Jessica Purkiss 15 Civilians Killed In Single ...
4        Howard Portnoy Iranian woman jailed for fictio...
                               ...                        
20795    Jerome Hudson Rapper T.I.: Trump a ’Poster Chi...
20796    Benjamin Hoffman N.F.L. Playoffs: Schedule, Ma...
20797    Michael J. de la Merced and Rachel Abrams Macy...
20798    Alex Ansary NATO, Russia To Hold Parallel Exer...
20799              David Swanson What Keeps the F-35 Alive
Name: content, Length: 20800, dtype: object


In [33]:
#separating the data and lable
X=df.drop(columns='label',axis=1)
Y=df['label']

In [34]:
print(X)
print(Y)

          id                                              title  \
0          0  House Dem Aide: We Didn’t Even See Comey’s Let...   
1          1  FLYNN: Hillary Clinton, Big Woman on Campus - ...   
2          2                  Why the Truth Might Get You Fired   
3          3  15 Civilians Killed In Single US Airstrike Hav...   
4          4  Iranian woman jailed for fictional unpublished...   
...      ...                                                ...   
20795  20795  Rapper T.I.: Trump a ’Poster Child For White S...   
20796  20796  N.F.L. Playoffs: Schedule, Matchups and Odds -...   
20797  20797  Macy’s Is Said to Receive Takeover Approach by...   
20798  20798  NATO, Russia To Hold Parallel Exercises In Bal...   
20799  20799                          What Keeps the F-35 Alive   

                                          author  \
0                                  Darrell Lucus   
1                                Daniel J. Flynn   
2                             Consortiu

stemming :

stemming is the process of reducing a word to its root word

In [35]:
port_stem=PorterStemmer()

In [36]:
def stemming(content):
  stemmed_content=re.sub('[^a-zA-Z]',' ',content)
  stemmed_content=stemmed_content.lower()
  stemmed_content=stemmed_content.split()
  stemmed_content=[port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content=' '.join(stemmed_content)
  return stemmed_content


In [37]:
df['content']=df['content'].apply(stemming)

In [38]:
df['text']=df['text'].apply(stemming)

In [39]:
X1=df['content'].values
Y=df['label'].values
X2=df['text'].values

In [40]:
print(X1)
print(X2)
print(Y)

['darrel lucu hous dem aid even see comey letter jason chaffetz tweet'
 'daniel j flynn flynn hillari clinton big woman campu breitbart'
 'consortiumnew com truth might get fire' ...
 'michael j de la merc rachel abram maci said receiv takeov approach hudson bay new york time'
 'alex ansari nato russia hold parallel exercis balkan'
 'david swanson keep f aliv']
['hous dem aid even see comey letter jason chaffetz tweet darrel lucu octob subscrib jason chaffetz stump american fork utah imag courtesi michael jolley avail creativ common licens apolog keith olbermann doubt worst person world week fbi director jame comey accord hous democrat aid look like also know second worst person well turn comey sent infam letter announc fbi look email may relat hillari clinton email server rank democrat relev committe hear comey found via tweet one republican committe chairmen know comey notifi republican chairmen democrat rank member hous intellig judiciari oversight committe agenc review email recent

In [41]:
#converting texual data to numerical data
vectorizer=TfidfVectorizer()
vectorizer.fit(X1)
X1=vectorizer.transform(X1)
vectorizer.fit(X2)
X2=vectorizer.transform(X2)

In [42]:
print(X1)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 210687 stored elements and shape (20800, 17128)>
  Coords	Values
  (0, 267)	0.2701012497770876
  (0, 2483)	0.36765196867972083
  (0, 2959)	0.24684501285337127
  (0, 3600)	0.3598939188262558
  (0, 3792)	0.27053324808454915
  (0, 4973)	0.23331696690935097
  (0, 7005)	0.2187416908935914
  (0, 7692)	0.24785219520671598
  (0, 8630)	0.2921251408704368
  (0, 8909)	0.36359638063260746
  (0, 13473)	0.2565896679337956
  (0, 15686)	0.2848506356272864
  (1, 1497)	0.2939891562094648
  (1, 1894)	0.15521974226349364
  (1, 2223)	0.3827320386859759
  (1, 2813)	0.19094574062359204
  (1, 3568)	0.26373768806048464
  (1, 5503)	0.7143299355715573
  (1, 6816)	0.1904660198296849
  (1, 16799)	0.30071745655510157
  (2, 2943)	0.3179886800654691
  (2, 3103)	0.46097489583229645
  (2, 5389)	0.3866530551182615
  (2, 5968)	0.3474613386728292
  (2, 9620)	0.49351492943649944
  :	:
  (20797, 3643)	0.2115550061362374
  (20797, 7042)	0.21799048897828685
  (2079

In [43]:
print(X2)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5077895 stored elements and shape (20800, 110429)>
  Coords	Values
  (0, 322)	0.04645169842778623
  (0, 517)	0.0245517461050009
  (0, 630)	0.04135372634373931
  (0, 867)	0.01681787047240059
  (0, 916)	0.01751633399794022
  (0, 1288)	0.02253182667968428
  (0, 1597)	0.019199506412857687
  (0, 1870)	0.1030987206166337
  (0, 2998)	0.04982130650611652
  (0, 3018)	0.02030766589984893
  (0, 3353)	0.012636792658322089
  (0, 3711)	0.03298297015308806
  (0, 4121)	0.017836647906280638
  (0, 4222)	0.020764429188216547
  (0, 4267)	0.028348204418323024
  (0, 4547)	0.018904895860127897
  (0, 4728)	0.027715985257229073
  (0, 4778)	0.04449451662092937
  (0, 4795)	0.015381707775372717
  (0, 6362)	0.019491218227923244
  (0, 6904)	0.022060732478280883
  (0, 8479)	0.02311183075491885
  (0, 8989)	0.015232435947664612
  (0, 10746)	0.050793141979697
  (0, 12131)	0.029779187109144727
  :	:
  (20799, 105823)	0.03677252622782508
  (20799, 105956)	0.04

In [44]:
X1_train,X1_test,Y1_train,Y1_test=train_test_split(X1,Y,test_size=0.2,stratify=Y,random_state=2)

In [45]:
X2_train,X2_test,Y2_train,Y2_test=train_test_split(X2,Y,test_size=0.2,stratify=Y,random_state=2)

In [46]:
model=LogisticRegression()
model.fit(X1_train,Y1_train)

LogisticRegression()

In [47]:
X1_train_prediction=model.predict(X1_train)
training_data_accuracy=accuracy_score(X1_train_prediction,Y1_train)
print('Accuracy score of the training data : ', training_data_accuracy)

Accuracy score of the training data :  0.9863581730769231


In [48]:
X1_test_prediction=model.predict(X1_test)
test_data_accuracy=accuracy_score(X1_test_prediction,Y1_test)
print('Accuracy score of the test data : ', test_data_accuracy)

Accuracy score of the test data :  0.9790865384615385


In [49]:
model.fit(X2_train,Y2_train)

LogisticRegression()

In [50]:
X2_train_prediction=model.predict(X2_train)
training_data_accuracy=accuracy_score(X2_train_prediction,Y2_train)
print('Accuracy score of the training data : ', training_data_accuracy)

Accuracy score of the training data :  0.9713341346153846


In [51]:
X2_test_prediction=model.predict(X2_test)
test_data_accuracy=accuracy_score(X2_test_prediction,Y2_test)
print('Accuracy score of the test data : ', test_data_accuracy)

Accuracy score of the test data :  0.9389423076923077
